# US-4: Phase 1 - URL Discovery

This notebook discovers all detail page URLs from sgcarmart.com listing pages.

**Phase 1 Only:** URL Discovery (crawls listing pages to extract detail URLs)

**Phase 2:** Handled in `05_workers_binding_crawl.ipynb` (crawls detail pages using Workers Bindings)

**Workflow:**
1. Run this notebook to discover all URLs
2. Run `05_workers_binding_crawl.ipynb` to crawl detail pages

## 1. Setup & Imports

In [36]:
import httpx
import json
import re
import os
import time
import sqlite3
import hashlib
from datetime import datetime
from tqdm import tqdm

print(f"URL Discovery started at: {datetime.now().isoformat()}")

URL Discovery started at: 2026-03-24T12:15:08.446721


In [37]:
# Load environment variables
with open(".env") as f:
    for line in f:
        if "=" in line:
            key, value = line.strip().split("=", 1)
            os.environ[key] = value

ACCOUNT_ID = os.environ["CLOUDFLARE_ACCOUNT_ID"]
API_TOKEN = os.environ["CLOUDFLARE_API_TOKEN"]
BASE_URL = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering"

print(f"Account ID: {ACCOUNT_ID[:8]}...{ACCOUNT_ID[-4:]}")

Account ID: 5decc977...c248


## 2. API Helper Functions

In [38]:
def get_headers():
    """Return authorization headers for Cloudflare API."""
    return {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }


def fetch_page_content(url: str, timeout: int = 120) -> tuple[str, float]:
    """Fetch page content using Cloudflare Browser Rendering /content endpoint.
    
    This endpoint respects query parameters (unlike /crawl which normalizes URLs).
    Used for listing pages with pagination.
    
    Returns tuple of (html, browser_seconds).
    """
    payload = {
        "url": url,
        "gotoOptions": {
            "waitUntil": "networkidle2",
            "timeout": 60000
        }
    }
    
    with httpx.Client(timeout=timeout) as client:
        response = client.post(f"{BASE_URL}/content", headers=get_headers(), json=payload)
        result = response.json()
    
    if not result.get("success"):
        raise Exception(f"Content fetch failed: {result}")
    
    html = result.get("result", "")
    # Note: /content endpoint doesn't return browser seconds, estimate ~5s per page
    return html, 5.0


print("API helper functions defined.")

API helper functions defined.


## 3. URL Extraction Functions

In [39]:
def extract_all_rsc_content(html: str) -> str:
    """Extract and combine all RSC push content from HTML."""
    push_pattern = r'self\.__next_f\.push\(\[(\d+),\s*"(.+?)"\]\)'
    push_matches = re.findall(push_pattern, html, re.DOTALL)
    
    all_content = ""
    for index_str, content in push_matches:
        try:
            unescaped = content.encode().decode('unicode_escape')
            all_content += unescaped
        except:
            all_content += content.replace('\\"', '"').replace('\\n', '\n')
    
    return all_content


def normalize_url(url: str) -> str:
    """Normalize URL to base path without query parameters.
    
    Examples:
    - /used-cars/info/car-123/?dl=3048 -> /used-cars/info/car-123/
    - /used-cars/info/car-123/?dl=3048&utm_content=SLeligible -> /used-cars/info/car-123/
    """
    # Remove query parameters
    if '?' in url:
        url = url.split('?')[0]
    # Ensure trailing slash for consistency
    if not url.endswith('/'):
        url += '/'
    return url


def extract_detail_urls_from_listing(html: str) -> list:
    """Extract detail page URLs from a listing page HTML."""
    all_content = extract_all_rsc_content(html)
    
    # Find all /used-cars/info/ URLs with their path
    # Pattern matches: /used-cars/info/car-name-123/?params followed by quote
    url_pattern = r'/used-cars/info/([^"\']+)'
    matches = re.findall(url_pattern, all_content)
    
    # Normalize to absolute URLs and deduplicate
    absolute_urls = set()
    for path in matches:
        # Clean up the path
        path = path.rstrip('"').rstrip("'")
        url = f"https://www.sgcarmart.com/used-cars/info/{path}"
        # Normalize to base path
        normalized = normalize_url(url)
        absolute_urls.add(normalized)
    
    return list(absolute_urls)


print("URL extraction functions defined.")

URL extraction functions defined.


In [40]:
class URLSeenStore:
    """Persistent storage for seen URLs using SQLite."""
    
    def __init__(self, db_path: str = "output/seen_urls.db"):
        self.conn = sqlite3.connect(db_path)
        self._init_db()
    
    def _init_db(self):
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS seen_urls (
                url_hash TEXT PRIMARY KEY,
                url TEXT NOT NULL,
                first_seen TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_url ON seen_urls(url)")
        self.conn.commit()
    
    def is_seen(self, url: str) -> bool:
        cursor = self.conn.execute(
            "SELECT 1 FROM seen_urls WHERE url = ? LIMIT 1", (url,)
        )
        return cursor.fetchone() is not None
    
    def mark_seen(self, url: str):
        self.conn.execute(
            "INSERT OR IGNORE INTO seen_urls (url_hash, url) VALUES (?, ?)",
            (hashlib.sha256(url.encode()).hexdigest(), url)
        )
        self.conn.commit()
    
    def mark_seen_batch(self, urls: list):
        """Batch insert for efficiency."""
        self.conn.executemany(
            "INSERT OR IGNORE INTO seen_urls (url_hash, url) VALUES (?, ?)",
            [(hashlib.sha256(u.encode()).hexdigest(), u) for u in urls]
        )
        self.conn.commit()
    
    def get_all_urls(self) -> set:
        """Load all seen URLs into memory."""
        cursor = self.conn.execute("SELECT url FROM seen_urls")
        return {row[0] for row in cursor.fetchall()}
    
    def get_count(self) -> int:
        cursor = self.conn.execute("SELECT COUNT(*) FROM seen_urls")
        return cursor.fetchone()[0]
    
    def close(self):
        self.conn.close()


print("URLSeenStore class defined.")

URLSeenStore class defined.


### Persistent URL Storage

The `URLSeenStore` class provides SQLite-backed storage for seen URLs:

- **1ms lookups** vs 50-100ms network calls (50-100x faster than cloud storage)
- **Zero cost** (Python stdlib, no API limits)
- **Offline capable** (no network dependency)
- **Incremental re-runs**: Skip already-discovered URLs

**How it works:**
1. First run: Discovers all ~14,000 URLs (takes ~1 hour)
2. Re-runs: Only checks for NEW listings (completes in ~1-2 minutes)

---

## Phase 1: URL Discovery

Crawl listing pages to extract all detail page URLs.

In [41]:
# Phase 1 Configuration
PHASE1_URLS_FILE = "output/all_detail_urls.json"
PHASE1_CHECKPOINT_FILE = "output/phase1_checkpoint.json"
SEEN_URLS_DB = "output/seen_urls.db"

# Pagination settings
MAX_LISTING_PAGES = 150 # Maximum pages to crawl 
LISTINGS_PER_PAGE = 100   # Approximate listings per page

# Early termination: Stop after N consecutive pages with no new URLs
MAX_EMPTY_PAGES = 3

# PRODUCTION MODE: Set MAX_LISTING_PAGES = 700 for full run
# TEST MODE: Uncomment the line below for testing with 5 pages only
# MAX_LISTING_PAGES = 5  

# Initialize persistent store
store = URLSeenStore(SEEN_URLS_DB)
existing_count = store.get_count()

print(f"Phase 1 - URL Discovery (Incremental)")
print(f"Existing URLs in database: {existing_count:,}")
print(f"Max pages: {MAX_LISTING_PAGES}")
print(f"Early termination: {MAX_EMPTY_PAGES} consecutive empty pages")

Phase 1 - URL Discovery (Incremental)
Existing URLs in database: 13,849
Max pages: 150
Early termination: 3 consecutive empty pages


In [42]:
# Phase 1: Incremental URL Discovery with early termination
print("=== Phase 1: Incremental URL Discovery ===")
print("Using /content endpoint with SQLite persistence\n")

# Load existing URLs from database
all_detail_urls = store.get_all_urls()
print(f"Loaded {len(all_detail_urls):,} existing URLs from database")

if len(all_detail_urls) == 0:
    print(f"\nFresh crawl: discovering all URLs from {MAX_LISTING_PAGES} pages...")
else:
    print(f"\nIncremental crawl: checking for new listings...")

new_urls_this_run = 0
pages_with_no_new_urls = 0
total_browser_seconds = 0
pages_crawled = 0
pages_failed = 0
page_results = {}

# Fetch listing pages with early termination
for page_num in tqdm(range(1, MAX_LISTING_PAGES + 1), desc="Crawling listing pages"):
    if page_num == 1:
        url = "https://www.sgcarmart.com/used-cars/listing?limit=100"
    else:
        url = f"https://www.sgcarmart.com/used-cars/listing?limit=100&page={page_num}"
    
    try:
        html, browser_seconds = fetch_page_content(url)
        
        if html:
            page_urls = extract_detail_urls_from_listing(html)
            
            # Filter to only NEW URLs (not in database)
            new_urls = [u for u in page_urls if not store.is_seen(u)]
            
            if not new_urls:
                pages_with_no_new_urls += 1
                page_results[page_num] = {
                    "url": url,
                    "status": "no_new_urls",
                    "urls_found": 0,
                    "total_on_page": len(page_urls)
                }
                
                # Early termination check
                if pages_with_no_new_urls >= MAX_EMPTY_PAGES:
                    print(f"\n\n⏹️ Early termination: {MAX_EMPTY_PAGES} consecutive pages with no new URLs")
                    break
            else:
                pages_with_no_new_urls = 0  # Reset counter
                new_urls_this_run += len(new_urls)
                
                # Add to in-memory set and persistent store
                all_detail_urls.update(new_urls)
                store.mark_seen_batch(new_urls)
                
                page_results[page_num] = {
                    "url": url,
                    "status": "completed",
                    "new_urls": len(new_urls),
                    "total_on_page": len(page_urls),
                    "html_length": len(html)
                }
            
            pages_crawled += 1
            total_browser_seconds += browser_seconds
        else:
            pages_failed += 1
            page_results[page_num] = {"url": url, "status": "empty_html"}
            
    except Exception as e:
        pages_failed += 1
        page_results[page_num] = {"url": url, "status": "error", "error": str(e)}
    
    # Periodic checkpoint save (every 50 pages)
    if pages_crawled % 50 == 0 and pages_crawled > 0:
        with open(PHASE1_URLS_FILE, 'w') as f:
            json.dump(list(all_detail_urls), f, indent=2)
        print(f"\n  Checkpoint: {len(all_detail_urls):,} total URLs, {new_urls_this_run} new this run")
    
    # Small delay between requests
    time.sleep(0.2)

# Convert to list for export
ALL_DETAIL_URLS = list(all_detail_urls)

# Save final results
with open(PHASE1_URLS_FILE, 'w') as f:
    json.dump(ALL_DETAIL_URLS, f, indent=2)

# Save page results
with open("output/phase1_page_results.json", 'w') as f:
    json.dump(page_results, f, indent=2)

# Save checkpoint info
phase1_checkpoint = {
    "timestamp": datetime.now().isoformat(),
    "max_pages": MAX_LISTING_PAGES,
    "pages_crawled": pages_crawled,
    "pages_failed": pages_failed,
    "new_urls_this_run": new_urls_this_run,
    "total_urls": len(ALL_DETAIL_URLS),
    "browser_seconds": total_browser_seconds,
    "early_terminated": pages_with_no_new_urls >= MAX_EMPTY_PAGES,
    "endpoint": "/content"
}
with open(PHASE1_CHECKPOINT_FILE, 'w') as f:
    json.dump(phase1_checkpoint, f, indent=2)

print(f"\n✅ Phase 1 Complete:")
print(f"   Pages crawled: {pages_crawled}/{MAX_LISTING_PAGES}")
print(f"   Pages failed: {pages_failed}")
print(f"   New URLs found: {new_urls_this_run}")
print(f"   Total URLs: {len(ALL_DETAIL_URLS):,}")
print(f"   Browser time: ~{total_browser_seconds:.0f}s ({total_browser_seconds/60:.1f} min)")
if phase1_checkpoint["early_terminated"]:
    print(f"   🚀 Early termination saved time!")

print(f"\n💾 Saved {len(ALL_DETAIL_URLS):,} URLs to output/all_detail_urls.json")
print(f"💾 SQLite database: output/seen_urls.db ({store.get_count():,} records)")

=== Phase 1: Incremental URL Discovery ===
Using /content endpoint with SQLite persistence

Loaded 13,849 existing URLs from database

Incremental crawl: checking for new listings...


Crawling listing pages:  13%|█▎        | 20/150 [03:05<20:07,  9.29s/it]



⏹️ Early termination: 3 consecutive pages with no new URLs

✅ Phase 1 Complete:
   Pages crawled: 20/150
   Pages failed: 0
   New URLs found: 403
   Total URLs: 14,252
   Browser time: ~100s (1.7 min)
   🚀 Early termination saved time!

💾 Saved 14,252 URLs to output/all_detail_urls.json
💾 SQLite database: output/seen_urls.db (14,252 records)


In [43]:
# Phase 1 Summary
print("=== Phase 1 Summary ===")
print(f"Total detail URLs in database: {store.get_count():,}")
print(f"Total detail URLs in export: {len(ALL_DETAIL_URLS):,}")
print(f"\nSample URLs:")
for url in ALL_DETAIL_URLS[:5]:
    print(f"  {url}")
if len(ALL_DETAIL_URLS) > 5:
    print(f"  ... ({len(ALL_DETAIL_URLS) - 5:,} more)")

# Load checkpoint info if available
if os.path.exists(PHASE1_CHECKPOINT_FILE):
    with open(PHASE1_CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    print(f"\nLast run info:")
    print(f"  Pages crawled: {ckpt.get('pages_crawled', 'N/A')}")
    print(f"  New URLs found: {ckpt.get('new_urls_this_run', 'N/A')}")
    print(f"  Browser time: {ckpt.get('browser_seconds', 0):.0f}s")
    if ckpt.get('early_terminated'):
        print(f"  🚀 Early termination: YES (saved ~1 hour)")

# Close database connection
store.close()
print("\n✅ Database connection closed")

=== Phase 1 Summary ===
Total detail URLs in database: 14,252
Total detail URLs in export: 14,252

Sample URLs:
  https://www.sgcarmart.com/used-cars/info/mercedes-benz-sl-class-sl350-coe-1403906/
  https://www.sgcarmart.com/used-cars/info/renault-grand-scenic-diesel-1480645/
  https://www.sgcarmart.com/used-cars/info/honda-civic-16a-vti-1484749/
  https://www.sgcarmart.com/used-cars/info/ferrari-california-43a-coe-1352719/
  https://www.sgcarmart.com/used-cars/info/toyota-corolla-altis-hybrid-1478249/
  ... (14,247 more)

Last run info:
  Pages crawled: 20
  New URLs found: 403
  Browser time: 100s
  🚀 Early termination: YES (saved ~1 hour)

✅ Database connection closed


---

## Next Steps

Run `05_workers_binding_crawl.ipynb` to crawl detail pages and extract structured data.

**Phase 2 in 05_workers_binding_crawl.ipynb:**
- Uses Cloudflare Workers Bindings (Puppeteer)
- 10 concurrent browsers
- ~40 URLs/min
- Incremental (skips already crawled URLs)

**Incremental URL Discovery:**
- Re-running this notebook will only collect NEW listings
- Early termination after 10 consecutive pages with no new URLs
- Re-runs complete in ~1-2 minutes instead of ~1 hour